In [1]:
# --- Dataerai provenance (optional) -----------------------------------------
# Traces this notebook run - cell source, outputs, logs, transfers, and
# environment - to the Dataerai platform when the SDK and daemon are
# available. Without them the notebook runs exactly as before.
try:
    %load_ext dataerai.magics
    %dataerai --trace --notebook MoS2_ptycho.ipynb abTEM / notebook runs / articles
except Exception as _dataerai_error:
    print(f"Dataerai tracing not active: {_dataerai_error}")

Signed in as demo@dataerai.com
Dataerai destination: abTEM / notebook runs / articles
Tracing notebook run e97e38b6-abd5-44fd-8724-3c08ac95781c. Cell source, outputs, logs, transfers, and environment details will be uploaded when %dataerai --finish runs.


# 4D-STEM of MoS2 including a ptychographic phase reconstruction

> **Note (2026):** this notebook was modernized from the abTEM 1.0-beta API it
> was published with to current abTEM 1.x: measurements save as Zarr instead of
> the removed HDF5 writer, noise/virtual detectors/center-of-mass are methods on
> `DiffractionPatterns`, and the `epie` function became the
> `RegularizedPtychographicOperator`. Physical parameters are unchanged, except
> that the chromatic focal-spread envelope is omitted (partial temporal
> coherence is configured differently in modern abTEM). The original code is
> preserved in the git history.

In [2]:
# 4D-STEM of MoS2 including a ptychographic phase reconstruction.
import matplotlib.pyplot as plt
import numpy as np
from ase.build import mx2

import abtem
from abtem import (
    GridScan,
    PixelatedDetector,
    Potential,
    Probe,
    dataerai,
    from_zarr,
    orthogonalize_cell,
    show_atoms,
)

abtem.config.set({"diagnostics.progress_bar": False})

# Create hexagonal supercell of MoS2
structure = mx2(formula='MoS2', kind='2H', a=3.18, thickness=3.127,
                size=(1, 1, 1), vacuum=10)
structure.pbc = True

# Orthogonalize cell and rotate and repeat it to match experiment.
atoms = orthogonalize_cell(structure)
atoms.rotate(-90, 'z', rotate_cell=True)
atoms *= (4, 4, 1)

# Switching x and y cell axes and centering to get positive coordinates.
atoms.cell = [[22.031686, 0, 0], [0, 12.72, 0], [0.0, 0.0, 12]]
atoms.center()

# Create S vacancy.
del atoms[64]

show_atoms(atoms);

In [3]:
# Create independent atom potential for the system.
potential = Potential(atoms, sampling=0.04)

# Experimental-style aberrations.
E = 80000
Csval = 3e5

# Probe convergence semi-angle.
alpha = 21.4

# Create scanning probe with partly guessed but reasonable parameters
# (astigmatism included; the chromatic focal-spread envelope of the original
# notebook is omitted in this modernization).
probe = Probe(
    energy=E,
    semiangle_cutoff=alpha,
    defocus=0,
    Cs=Csval,
    C12=10,
    phi12=2.0,
)

In [4]:
exp = dataerai.start_run(
    name="MoS2 4D-STEM ptychography",
    collection="abTEM / notebook runs / articles",
)
exp.capture_structure(atoms)
exp.capture_potential(potential)
exp.capture_illumination(probe)

'illumination-probe'

In [5]:
# Define the scan with experimental scan sampling.
scanstep1 = 0.21
scanstep2 = 0.45
scanstart = [0, 0]
scanend = [atoms.cell[0][0], atoms.cell[1][1]]

gridscan1 = GridScan(start=scanstart, end=scanend, sampling=scanstep1)
gridscan2 = GridScan(start=scanstart, end=scanend, sampling=scanstep2)

# Define a pixelated 4D detector.
pixelated_detector = PixelatedDetector(max_angle=5 * alpha)

In [6]:
exp.capture_detector(pixelated_detector)

'detector-pixelateddetector'

In [7]:
# Running the multislice simulations takes around 10 minutes on a fast CPU.
exp.capture_scan(gridscan1)
measurements = probe.scan(
    potential, scan=gridscan1, detectors=pixelated_detector
)
measurements.to_zarr('iam_mos2_0.21.zarr.zip', overwrite=True)

exp.capture_scan(gridscan2)
measurements_ptycho = probe.scan(
    potential, scan=gridscan2, detectors=pixelated_detector
)
measurements_ptycho.to_zarr('iam_mos2_0.45.zarr.zip', overwrite=True)

'iam_mos2_0.45.zarr.zip'

In [8]:
# Reading saved measurements from disk.
measurements = from_zarr('iam_mos2_0.21.zarr.zip').compute()
measurements_ptycho = from_zarr('iam_mos2_0.45.zarr.zip').compute()

In [9]:
# Add Poisson noise to the 4D measurement, and integrate for BF, ADF and iCOM.
noisy = measurements.poisson_noise(dose_per_area=3 * 10**5, seed=13)

bf = noisy.integrate_radial(0, alpha)
bf_diff = bf.diffractograms()

adf = noisy.integrate_radial(3 * alpha, 4 * alpha)
adf_diff = adf.diffractograms()

icom = noisy.integrated_center_of_mass()
icom_diff = icom.diffractograms()

# Preserve the derived virtual-detector images in the provenance run.
exp.capture_measurement(bf, name='bright_field')
exp.capture_measurement(adf, name='annular_dark_field')
exp.capture_measurement(icom, name='integrated_center_of_mass')


'measurement-integrated-center-of-mass'

In [10]:
# Bandlimit and add Poisson noise to the coarser 4D measurement, then
# reconstruct the phase with the regularized PIE algorithm.
from abtem.reconstruct import RegularizedPtychographicOperator

limited = measurements_ptycho.bandlimit(0, 5 * alpha)
noisy_ptycho = limited.poisson_noise(dose_per_area=3 * 10**5, seed=13)

# The initial probe guess is aberration-free, as in the original notebook.
operator = RegularizedPtychographicOperator(
    noisy_ptycho,
    energy=E,
    semiangle_cutoff=alpha,
    device='cpu',
    preprocess=True,
)

objects, probes, positions, errors = operator.reconstruct(
    max_iterations=5,
    return_iterations=True,
    fix_com=True,
    random_seed=13,
    pre_probe_correction_update_steps=10**9,  # keep the probe fixed
)

In [11]:
# The last iteration of the reconstructed object; its phase is the image.
final_object = objects[-1]
ptycho = (
    final_object.angle()
    if hasattr(final_object, 'angle')
    else abtem.Images(np.angle(np.asarray(final_object.array)), sampling=final_object.sampling)
)
ptycho_diff = ptycho.diffractograms()

# Preserve the ptychographic phase reconstruction (Fig. 2d).
exp.capture_measurement(ptycho, name='ptychographic_phase')

ptycho.show();

In [12]:
# Plotting a figure to reproduce Figure 2 of doi:10.1038/s41586-018-0298-5.
# Note that the simulation was done for the periodic supercell, but the plotted
# areas are slightly cropped below to match the experimental field of view.
fig, axes = plt.subplots(4, 2, figsize=(16, 18))

for ax in axes.flat:
    ax.set_axis_off()

xlim = [2.7, atoms.cell[0][0] - 1.0]
ylim = [0, atoms.cell[1][1] - 1.6]

bf.show(ax=axes[0, 0], title='(a) Bright-field')
axes[0, 0].set_xlim(xlim); axes[0, 0].set_ylim(ylim)
bf_diff.show(ax=axes[0, 1], power=0.15, cmap='afmhot', title='(e)')

adf.show(ax=axes[1, 0], title='(b) ADF')
axes[1, 0].set_xlim(xlim); axes[1, 0].set_ylim(ylim)
adf_diff.show(ax=axes[1, 1], power=0.15, cmap='afmhot', title='(f)')

icom.show(ax=axes[2, 0], title='(c) iCoM')
axes[2, 0].set_xlim(xlim); axes[2, 0].set_ylim(ylim)
icom_diff.show(ax=axes[2, 1], power=0.15, cmap='afmhot', title='(g)')

ptycho.show(ax=axes[3, 0], title='(d) Ptychography')
ptycho_diff.show(ax=axes[3, 1], power=0.15, cmap='afmhot', title='(h)')

plt.tight_layout()

In [13]:
dataerai.finish_run()

In [14]:
# --- Dataerai provenance: publish the execution trace, if one is active -----
try:
    %dataerai --finish
except Exception as _dataerai_error:
    print(f"Dataerai trace not published: {_dataerai_error}")

Notebook trace will publish after this cell finishes.


Published notebook execution trace e97e38b6-abd5-44fd-8724-3c08ac95781c (13 cells, 13 products).
